* Instalação dos pacotes inexistentes

In [ ]:
!pip install pypdf2

* Importação das bibliotecas

In [ ]:
import PyPDF2
import re
import pandas as pd


* começar a programar

In [ ]:
pdf_file = open('extrato 29-08-2026.pdf','rb')

# Leitura usando o PdfReader
pdf = PyPDF2.PdfReader(pdf_file)

# Numero de paginas
paginas = len(pdf.pages)

paginas

3

In [ ]:
# Le e todas as paginas
paginas_pdf = pdf.pages

# Inicializa uma string vazia para armazenar todo o texto
texto = ""

# Itera sobre cada pagina e extrai o texto
for i in range(len(paginas_pdf)):
    texto += paginas_pdf[i].extract_text()

# Imprime o texto
print(texto)

Solicitado em: 29/08/2026 - 16h43
RHUAN FRANCISCO DE AQUINO
CPF/CNPJ: 144.558.877-33, Instituição: Banco Inter, Agência: 0001-9, Conta: 22954743-5
Período: 29/07/2026 a 29/08/2026
Saldo total
R$ 1,86
(bloqueado + disponível)Saldo disponível:
R$ 1,86Saldo bloqueado:
R$ 0,00
Valor Saldo por transação 29 de Julho de 2026 Saldo do dia: R$ 276,31
Compra no debito: "No estabelecimento DOCES GUIMARAES CACHOEIRAS DE BRA" -R$ 6,50 R$ 286,80
Compra no debito: "No estabelecimento MERCADO VIDAL LTDA CACHOEIRAS DE BRA" -R$ 10,49 R$ 276,31
30 de Julho de 2026 Saldo do dia: R$ 257,87
Compra no debito: "No estabelecimento MERCADO VIDAL LTDA CACHOEIRAS DE BRA" -R$ 18,44 R$ 257,87
31 de Julho de 2026 Saldo do dia: R$ 225,54
Pix recebido: "Cp :18236120-Barbara da Silva Medeiros" R$ 23,50 R$ 281,37
Pix enviado: "Cp :19468242-IFOODCOM AGENCIA DE RESTAURANTES ONLINE SA" -R$ 47,24 R$ 234,13
Compra no debito: "No estabelecimento MERCADO VIDAL LTDA CACHOEIRAS DE BRA" -R$ 8,59 R$ 225,54
1 de Agosto de 2026 Sald

In [ ]:

# Texto extraído da página do PDF
linhas = texto.strip().split("\n")
data_atual = ""
lista_dados = []

# Loop para analisar linha por linha do texto
for line in linhas:
  # Identifica a linha de data
  data_match = re.search(r'(\d{1,2}\s+de\s+[A-Za-zç]+\s+de\s+\d{4})', line)
  if data_match and 'Saldo do dia' in line:
    data_atual = data_match.group(1)
    continue

  # Identifica as linhas de transação
  if ":" in line and ('R$' in line or '-R$' in line):
    parts = line.split(': "', 1)
    if len(parts) == 2:
      t_type = parts[0].strip()
      rest = parts[1]
      desc_end = rest.rfind('"')
      if desc_end != -1:
        t_desc = rest[:desc_end]
        values_part = rest[desc_end + 1 :].strip()
        currencies = re.findall(r"-?R\$\s*[\d\.,]+", values_part)
        if len(currencies) >= 2:
          lista_dados.append({
              'Data': data_atual,
              'Tipo': t_type,
              'Descrição': t_desc,
              'Valor (R$)': currencies[0],
              'Saldo Após (R$)': currencies[1],
          })

# Cria o DataFrame
df = pd.DataFrame(lista_dados)
print(df.to_markdown(index=False))

| Data                 | Tipo                                                                                                               | Descrição                                                   | Valor (R$)   | Saldo Após (R$)   |
|:---------------------|:-------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------|:-------------|:------------------|
| 29 de Julho de 2026  | Compra no debito                                                                                                   | No estabelecimento DOCES GUIMARAES CACHOEIRAS DE BRA        | -R$ 6,50     | R$ 286,80         |
| 29 de Julho de 2026  | Compra no debito                                                                                                   | No estabelecimento MERCADO VIDAL LTDA CACHOEIRAS DE BRA     | -R$ 10,49    | R$ 276,31         |
| 30 de Julho de 2026  | Compra no debito   

In [ ]:
# Dataframe de teste
df_teste = pd.DataFrame(df)

In [ ]:
padrao_para_remover = (
    "SAC: 0800 940 9999 (opção 09) Ouvidoria: 0800 940 7772 Deﬁciência de"
    " fala e audição: 0800 979 7099"
)

# Busca o padrão que será removido, retirando assim todo o texto do SAC desnecessario
df['Tipo'] = (
    df['Tipo'].str.replace(padrao_para_remover, '', regex=False).str.strip()
)

padrao_remocao2 = (
    r'No estabelecimento |Cp :\d+-'  # Remove "No estabelecimento " OU "Cp :[números]-"
)

# Busca e remove texto desnecessario da coluna 'Descrição'
df['Descrição'] = df['Descrição'].str.replace(
    padrao_remocao2, '', regex=True
)

df

,Data,Tipo,Descrição,Valor (R$),Saldo Após (R$)
0,29 de Julho de 2026,Compra no debito,DOCES GUIMARAES CACHOEIRAS DE BRA,"-R$ 6,50","R$ 286,80"
1,29 de Julho de 2026,Compra no debito,MERCADO VIDAL LTDA CACHOEIRAS DE BRA,"-R$ 10,49","R$ 276,31"
2,30 de Julho de 2026,Compra no debito,MERCADO VIDAL LTDA CACHOEIRAS DE BRA,"-R$ 18,44","R$ 257,87"
3,31 de Julho de 2026,Pix recebido,Barbara da Silva Medeiros,"R$ 23,50","R$ 281,37"
4,31 de Julho de 2026,Pix enviado,IFOODCOM AGENCIA DE RESTAURANTES ONLINE SA,"-R$ 47,24","R$ 234,13"
5,31 de Julho de 2026,Compra no debito,MERCADO VIDAL LTDA CACHOEIRAS DE BRA,"-R$ 8,59","R$ 225,54"
6,1 de Agosto de 2026,Compra no debito,MERCADO VIDAL LTDA CACHOEIRAS DE BRA,"-R$ 24,41","R$ 201,13"
7,4 de Agosto de 2026,Compra no debito,MERCADO VIDAL LTDA CACHOEIRAS DE BRA,"-R$ 28,99","R$ 172,14"
8,6 de Agosto de 2026,Compra no debito,LANCHONETE PONTO CERTO CACHOEIRAS DE BRA,"-R$ 5,00","R$ 167,14"
9,6 de Agosto de 2026,Compra no debito,MERCADO VIDAL LTDA CACHOEIRAS DE BRA,"-R$ 5,98","R$ 161,16"


In [ ]:
#Converte o Dataframe em um arquivo para Excel
df.to_excel('extrato_bancario.xlsx', index=False)
print('DataFrame salvo como extrato_bancario.xlsx')

DataFrame salvo como extrato_bancario.xlsx
